<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c42-header.png" alt="Nextia Learning · Vision AI: See, Count and Inspect" width="100%">

# Set the line with the warehouse

**[Set the line with the warehouse](https://learning.nextia-ai.com/courses/vision/m06/set-the-line-with-the-warehouse/)** · Vision AI: See, Count and Inspect · Module 6, lesson 3 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** turn anomaly scores into pass, check and fail with two lines, compute what a pair of lines costs per day, choose the cheapest lines for Hamid's budget on the validation photos, and see how much those lines rest on a few damaged photos.

| | |
|---|---|
| **Time** | About 30 minutes |
| **Needs** | An internet connection for the setup cells. No model download, no account and no key: the scores were recorded once by the course team. |
| **You should know** | Anomaly scores and AUROC, from [Anomaly-based inspection](https://learning.nextia-ai.com/courses/vision/m06/anomaly-based-inspection/); the cost of each mistake, from [The return that cost twice](https://learning.nextia-ai.com/courses/vision/m01/the-return-that-cost-twice/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1, Pillow 13.0.0, scikit-learn 1.9.1 and matplotlib 3.11.2 (PyTorch 2.15.0 and timm 1.0.31 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/vision/m06/set-the-line-with-the-warehouse/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C42/M06-L03-set-the-line-with-the-warehouse/set-the-line-with-the-warehouse-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, pillow 13.0.0, scikit-learn 1.9.1, scikit-image 0.26.0, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 pillow==13.0.0 scikit-learn==1.9.1 scikit-image==0.26.0 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
import numpy; print("numpy", numpy.__version__)
import pandas; print("pandas", pandas.__version__)
import PIL; print("pillow", PIL.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)
import skimage; print("scikit-image", skimage.__version__)

## Setup: get the Larkfield images

Run the next cell. It downloads the image sets that this lesson needs (about 19 MB, as zip files) into a folder `larkfield-images`, with the course's helper `compare.py`, checks each file's checksum and unpacks it. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-images` (in Colab: the **Files** panel on the left), then run the cell again.

Where the photos come from: the catalogue photos are cut from Open Images photos (CC BY 2.0: the author of each photo is in `data/catalogue/manifest.csv`), the returned candles are photos of the VisA inspection set (CC BY 4.0), and the pallets are drawn by our code. Read `data/dataset-card.md` for the details.

In [ ]:
# Setup: download the image sets into larkfield-images/, check them and unpack them.
import hashlib, os, shutil, subprocess, urllib.request, zipfile
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C42/"
FILES = {
    "archives/returns.zip": "dcf02f521f68228d34717b303628b2db8b19d18433126da2ce64047f1449210f",
    "archives/extras.zip": "f121137bc765866c25929a4233c557ba72b64eea0c806070cc04a86c28ec27ec",
    "compare.py": "65e58d2fe043105e3ff09626861a8f64e5548039df760ea10d6fc02c71f8fec3",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-images").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-images and run again.")
    if name.endswith(".zip"):
        with zipfile.ZipFile(path) as z:
            z.extractall(PROJECT / "data")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
n = sum(1 for p in Path("data").rglob("*") if p.is_file())
print(f"Ready: {n} files in larkfield-images/data.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, Pillow's `Image`, matplotlib, and the course helper as `cp` (the comparison table). It defines `show(paths)`, which draws photos in a row, and the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: images, the comparison table, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import compare as cp
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
CLASSES = ["bowl", "candle", "chair", "cushion", "lamp", "mug", "plant_pot", "vase"]

def show(paths, titles=None, size=2.2):
    """Draw photos in a row (at most 8)."""
    paths = list(paths)[:8]
    fig, axes = plt.subplots(1, len(paths), figsize=(size * len(paths), size))
    for i, (ax, p) in enumerate(zip(np.atleast_1d(axes), paths)):
        ax.imshow(Image.open(p)); ax.axis("off")
        if titles:
            ax.set_title(str(titles[i]), fontsize=8)
    plt.show()

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. A tiny example by hand

Ten returns, their scores, and two lines: **check from 0.4**, **fail from 0.6**. Below 0.4 the item passes and goes back on sale. From 0.6 up it fails and goes to the outlet. Between the lines, a person checks it, and a person is always right.

> **Predict.** How many items pass, how many are checked, and how many fail? Which mistakes are left?

In [ ]:
tiny = pd.DataFrame({"score":   [0.12, 0.25, 0.31, 0.38, 0.45, 0.52, 0.58, 0.63, 0.71, 0.90],
                     "damaged": [False, False, True, False, False, True, False, False, True, True]})
tiny["zone"] = cp.zones(tiny["score"], 0.4, 0.6)
print(tiny.to_string(index=False))
print(cp.inspection_costs(tiny["score"], tiny["damaged"], 0.4, 0.6))

One damaged item (0.31) passes: it is sold as new. One good item (0.63) fails: it goes to the outlet. Three items are checked. `inspection_costs` scales the ten items to Hamid's 600 returns a day: each item stands for 60 returns. So the day has 60 damaged items sold as new (60 x 45 = 2,700 dollars), 60 good items lost (60 x 28 = 1,680) and 180 checks (180 x 0.50 = 90): 4,470 dollars a day, and 180 checks is over the budget of 100.

## 2. The recorded scores of PatchCore

These are the scores of anomalib's PatchCore (wide ResNet-50) from the last lesson, recorded once by the course team, so every learner gets the same numbers.

In [ ]:
returns = pd.read_csv("data/returns/manifest.csv")
scores = json.load(open("data/predictions/returns_patchcore_wide_resnet50.json"))
val, test = (returns[returns["split"] == s].reset_index(drop=True) for s in ("val", "test"))
sv, st = val["item_id"].map(scores).values, test["item_id"].map(scores).values
dv, dt = (val["status"] == "damaged").values, (test["status"] == "damaged").values
print(f"validation: {len(val)} photos, {dv.sum()} damaged; test: {len(test)} photos, {dt.sum()} damaged")
print("AUROC on test:", cp.roc_auc(st, dt))

## 3. The cheapest lines for 100 checks a day

`cp.choose_lines` tries every pair of lines from 201 points of the validation scores. It keeps the pair with the lowest cost per day whose checks stay inside the budget. We choose on the **validation** photos, then measure once on the **test** photos.

In [ ]:
lines = cp.choose_lines(sv, dv)
print(f"check from {lines['check_line']:.4f}, fail from {lines['fail_line']:.4f}")
print("validation:", {k: lines[k] for k in ("checks_per_day", "damaged_sold_as_new_per_day", "good_marked_damaged_per_day", "cost_per_day")})
on_test = cp.inspection_costs(st, dt, lines["check_line"], lines["fail_line"])
print("test:      ", {k: on_test[k] for k in ("checks_per_day", "damaged_sold_as_new_per_day", "good_marked_damaged_per_day", "cost_per_day")})

> **Check.** Check from 0.4106, fail from 0.6441; 37.5 dollars a day on validation and 368.8 on test. The test cost is ten times the validation cost. Why? Keep the question for section 5.

## 4. Hamid's budget falls to 50 checks

> **Predict.** If Hamid's team can do only 50 checks a day, does the cost go up a little or a lot?

In [ ]:
half = dict(cp.COSTS, checks_per_day=50)
lines_50 = cp.choose_lines(sv, dv, half)
on_test_50 = cp.inspection_costs(st, dt, lines_50["check_line"], lines_50["fail_line"], half)
print(f"check from {lines_50['check_line']:.4f}, fail from {lines_50['fail_line']:.4f}")
print("test:", {k: on_test_50[k] for k in ("checks_per_day", "damaged_sold_as_new_per_day", "good_marked_damaged_per_day", "cost_per_day")})

A lot: 1272.9 dollars a day instead of 368.8. The check line stays, so the same damaged items pass. To halve the checks, the fail line comes down to 0.4859, into the good candles, and many more good items go to the outlet.

## 5. Twelve damaged photos hold the line

The validation set has only 12 damaged photos. Look at their scores, and at where the check line sits.

In [ ]:
print("damaged validation scores:", np.round(np.sort(sv[dv]), 3))
print("damaged test scores:      ", np.round(np.sort(st[dt]), 3))
print(f"check line {lines['check_line']:.4f}")

The check line sits just below the **lowest** damaged validation score (0.4122). One photo decides it. On the test photos, two damaged candles score lower (0.244 and 0.339: damaged corners of the packaging), so they pass.

What if that one photo had not been in the validation set? Remove it and choose again:

In [ ]:
lowest = np.where(dv)[0][np.argmin(sv[dv])]
keep = np.arange(len(sv)) != lowest
lines_wo = cp.choose_lines(sv[keep], dv[keep])
print(f"without {val['item_id'][lowest]}: check from {lines_wo['check_line']:.4f}, fail from {lines_wo['fail_line']:.4f}")
print("test cost:", cp.inspection_costs(st, dt, lines_wo["check_line"], lines_wo["fail_line"])["cost_per_day"])

One photo less moves the check line to 0.4219 and the test cost from 368.8 to 488.5 dollars a day. A line set on 12 examples is a guess. The course team measured the chosen lines once on a hidden test set of 135 photos (15 damaged, not in labs): **1,468.9 dollars a day**, because 6 of the 15 damaged candles passed. The [lesson page](https://learning.nextia-ai.com/courses/vision/m06/set-the-line-with-the-warehouse/) explains what Hamid does about it.

## 6. Your turn: a budget of 25 checks

> **Your turn.** Choose the lines on the validation photos for a budget of **25** checks a day, and put the **test** cost per day in `cost_25`. Use `dict(cp.COSTS, checks_per_day=25)`, `cp.choose_lines` and `cp.inspection_costs`, as in section 4.

In [ ]:
cost_25 = ...

In [ ]:
expect('cost_25', cost_25, 1727.1, tolerance=0.0005)

With 25 checks a day, the fail line comes down even further. No inspection at all costs 2,229.4 dollars a day on the same test photos, so a smaller budget eats most of what the inspector saves.

## Next

The [lesson page](https://learning.nextia-ai.com/courses/vision/m06/set-the-line-with-the-warehouse/) has the zones explorer, the hidden test and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Find the same product](https://learning.nextia-ai.com/courses/vision/m07/find-the-same-product/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/vision/m06/set-the-line-with-the-warehouse/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The photos are open photos (each with its author and licence in the manifests) or images made by our code; no photo shows a real person.